# MNIST classify with MLP

### (1) Data Process

이번에 다루는 데이터는 npy데이터이다. 

npy데이터는 csv파일을 numpy로 불러올때 사용하는 genfromtxt가 아닌 load 를 이용하여 불러온다.

In [4]:
import numpy as np 

data1 = np.load("./MNIST_data/data1.npy")
data2 = np.load("./MNIST_data/data2.npy")
data3 = np.load("./MNIST_data/data3.npy")
data4 = np.load("./MNIST_data/data4.npy")
data1.shape, data2.shape, data3.shape, data4.shape


((1000, 12, 12), (1000, 12, 12), (1000, 12, 12), (1000, 12, 12))

### (2) visuallize

mnist데이터가 3차원 데이터임을 확인하였다.

mnist데이터를 img로 불러오기 위한 시각화를 해보자.

# 3. MLP class

MLP class를 만들어보자.


    (1) 학습 데이터 생성

주어진 데이터의 쉐입은 (1000,12, 12)의 크기 데이터 4개이다. 

이를 학습데이터로 사용하기 위해서는 4000,144로 하나의 행렬로 만들어야한다.

이때 만들어진 4000,144가 신경망의 핵심이다.

주어진 데이터 그 자체를 학습하는것이 신경망이기 때문이다.

- n_sample에 샘플의 갯수를 저장하여 하드코딩을 방지하자.

- 12, 12 는 reshape(n, -1)을 통해 2차원 벡터로 변환하자.

- np.concatenate([vector1, vector2 ---], axis = (row = 0, col = 1))

    np.concatenate를 통해 axis로 병합할 방향과 벡터를 작성하자

    이때 row방향이라면 열의 갯수는 그대로 이다.



In [7]:
n_sample = data1.shape[0]

data1 = data1.reshape(n_sample, -1)
data2 = data2.reshape(n_sample, -1)
data3 = data3.reshape(n_sample, -1)
data4 = data4.reshape(n_sample, -1)

print(data1.shape, data2.shape, data3.shape, data4.shape)

feature_matrix = np.concatenate([data1, data2, data3, data4], axis = 0)
print(feature_matrix.shape)



(1000, 144) (1000, 144) (1000, 144) (1000, 144)
(4000, 144)


본격적으로 MLP class를 구성해보자.

MLP란 multy layer perceptron의 약자이다.

MLP는 입력 - 은닉 - 출력 으로 구성되어있는 형태이다.

# Layer

### (1) 입력층

입력층에서는 4000,144 데이터 자체를 입력받는다.

입력받은 데이터 4000,144의 데이터는 선형방정식 y = ax + b에 적용된다.

즉, a는 가중치인데, 4000,144와 곱이 성립되기 위해서 가중치의 shape을 맞춰주어야한다.

입력층 - 은닉층의 가중치를 w1이라하자.

### (2) 은닉층

은닉층은 y = f(ax+b)와 같은 입력 - 출력에서의 비선형적 한계를 보완해준다.

즉 여러 복잡한 패턴을 더 잘 학습하게 도와주는 중요한 역할을 한다.

이때 은닉층의 갯수와 활성화 함수는 1:1 이므로, 은닉층을 여러겹 쌓을수록 더 많은 활성화 함수를 사용가능하고,

그에 따라 복잡한 패턴도 인식하기 쉬워진다.

은닉층의 가중치 설정을 생각해보자.

4000,144의 입력데이터가 들어오고, 입력 - 은닉은 (144, x)의 shape일 것이다.

그리고 그 결합의 결과는 (4000,x) 가 될것인데, 이때 x는 은닉층의 뉴런(perceptron)의 갯수이다.

은닉층의 뉴런 갯수는 사용자가 직접 설정 가능한 부분으로, 만약 50개의 뉴런을 사용한다하면 4000,50이 될것이다.

### (3) 출력층

출력층은 은닉층에서 받은 데이터를 요약한다. 데이터가 4000,144로 들어왔다면, 4개의 class. 

즉 정답값인 1,2,3,4를 각각 할당하므로 (4000,4)의 shape을 반환해야한다.


---

주어진 정보를 바탕으로 MLP class를 작성해보자.

은닉층은 2개로 구성하고, 각 층은 50, 100개의 뉴런을 가진다고 가정하자.






In [ ]:
class MLP:
    def __init__(self, input_dim = 144, hidden1 = 50, hidden2 = 100, output_dim = 4):
        self.w1 = np.zeros((input_dim, hidden1))
        self.b1 = np.zeros((hidden1,))

        self.w2 = np.zeros((hidden1, hidden2))
        self.b2 = np.zeros((hidden2,))
        
        self.w3 = np.zeros((hidden2, output_dim))
        self.b3 = np.zeros((output_dim,))

# Activation function

다음은 활성화함수이다. 입력 - 은닉1 - 은닉2 - 출력 으로 데이터 전송되는 과정에서,

각 층에서 비선형적 특징을 표현하기 위해 활성화 함수를 사용한다.

대표적으로 

    -relu, softmax, sigmoid, tanh 과 같은 함수들이 존재한다.

여기에서는 softmax와 relu를 통해 실습한다.

    relu를 사용하는 이유는, 후에 역전파와 순전파 연산시 미분값이 깔끔하다는 이유이다.

### (1) relu

첫번째 입력: (4000,144) @ (144, 50) + (50,) == (4000,50) 을 가정하자.

4000,50은 첫번째 은닉층의 각 perceptron에 4000개의 샘플에 대한 정보값이 저장된다.

이때 relu함수에 (4000,50)의 결과를 대입하게되면, 음수 or 양수의 결과가 나오게되고,

양수라면 활성뉴런, 음수라면 비활성뉴런으로 파악한다. 

2번째 은닉층에서도 relu를 사용하여 (4000,100)의 데이터로 압축한다.

    - 2번째 은닉층에서 relu를 사용하지않고 다른 활성화 함수를 사용한다면? -

### (2) softmax

출력층에와서 양수 음수로 이루어진 4000,100의 데이터를 4000,4로 압축한다. 

이때 softmax를 사용했을때 4000,4의 데이터의 정보에 대해 알아보자.

4000,4의 각 행은 sample을 이야기한다. 

이때 column이 가진 정보는 1열은 숫자 1에 대한 확률, 2에대한 확률...

따라서 4개의 열은 각 label이 가진 확률 정보를 가지고 있다.

우리는 위 4000,4의 가장 큰값을 가진 열의 인덱스를 하나씩 뽑아 4000,1 로 만든 결과를 사용한다.

---

활성화 함수를 통해 4000,1 의 예측 결과를 만들었다.

정확도를 높이기 위해서 위 결과를 사용하여보자.

# forward propagation(순전파)

순전파는 위에서 우리가 각 층에서 활성화 함수들을 거쳐 도출해내는 과정을 이야기한다.

즉 예측 결과이다.

이때 정확도는 낮을것이다. 

이에 따라 정확도를 높이기 위해 가중치를 갱신하여야하고,

가중치를 갱신하는 과정을 역전파라한다.

# backward propagation(역전파)

도출된 결과를 통해 실제 정답과 예측 결과의 오차를 통해 가중치를 갱신하는 방법이다.

손실 값은 오차의 평균값이며, 손실을 가중치에 대해 grad를 구하여 가장 큰 방향의 역방향으로 움직이는 방식이다.

grad는 가장 가파르게 움직이는 벡터의 방향이고, 그 역방향은 오차를 최소로 줄이는 방향으로 해석된다.

역전파는 각 층에서 연산한다.

우리는 출력층에서 softmax, 은닉층1,2에서 relu함수를 사용하였으므로, softmax와 relu함수의 미분식을 사용한다.

이때 각 층에서의 모든 오차를 구해야한다. 

각 층에서 오차 평균을 구하고, 그 오차에 대한 가중치2에 의한 grad를 구한다.

후에 grad를 이용하여 각 층의 가중치를 갱신한다.



In [ ]:
def backward(self, X, y_true):
        m = X.shape[0]

        # 출력층 오차
        delta3 = (self.a3 - y_true) / m
        dw3 = self.a2.T @ delta3
        db3 = np.sum(delta3, axis=0)

        # 은닉2층 오차
        delta2 = (delta3 @ self.w3.T) * self.sigmoid_deriv(self.z2)
        dw2 = self.a1.T @ delta2
        db2 = np.sum(delta2, axis=0)

        # 은닉1층 오차
        delta1 = (delta2 @ self.w2.T) * self.sigmoid_deriv(self.z1)
        dw1 = X.T @ delta1
        db1 = np.sum(delta1, axis=0)

        # 파라미터 업데이트 (SGD)
        self.w3 -= self.lr * dw3
        self.b3 -= self.lr * db3
        self.w2 -= self.lr * dw2
        self.b2 -= self.lr * db2
        self.w1 -= self.lr * dw1
        self.b1 -= self.lr * db1